<a href="https://colab.research.google.com/github/vivekgautamgv/Python-For-Finance/blob/main/New_Strategy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance pandas numpy scikit-learn

In [34]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import StandardScaler

class InnovativeCryptoStrategy:
    def __init__(self, lookback_period=14):
        self.lookback_period = lookback_period
        self.scaler = StandardScaler()

    def calculate_order_book_imbalance(self, data):
        """
        Calculate order book imbalance from order book data
        """
        df = data.copy()
        df['imbalance'] = (df['buy_volume'] - df['sell_volume']) / (df['buy_volume'] + df['sell_volume'])
        return df

    def analyze_social_sentiment(self, data):
        """
        Analyze social sentiment
        """
        df = data.copy()
        df['sentiment_score'] = df['positive_mentions'] - df['negative_mentions']
        return df

    def analyze_blockchain_activity(self, data):
        """
        Analyze blockchain activity
        """
        df = data.copy()
        df['whale_alert'] = df['large_transactions'] > df['threshold']
        return df

    def generate_unique_signals(self, data):
        df = data.copy()
        df = self.calculate_order_book_imbalance(df)
        df = self.analyze_social_sentiment(df)
        df = self.analyze_blockchain_activity(df)

        df['position'] = 0

        for i in range(self.lookback_period, len(df)):
            imbalance_signal = df['imbalance'].iloc[i] if not pd.isna(df['imbalance'].iloc[i]) else 0
            sentiment_signal = df['sentiment_score'].iloc[i] if not pd.isna(df['sentiment_score'].iloc[i]) else 0
            whale_activity = df['whale_alert'].iloc[i] if not pd.isna(df['whale_alert'].iloc[i]) else 0

            combined_signal = (
                0.50 * np.sign(imbalance_signal) +
                0.35 * np.sign(sentiment_signal) +
                0.15 * whale_activity
            )

            if combined_signal > 0.5:
                df.loc[df.index[i], 'position'] = 1
            elif combined_signal < -0.5:
                df.loc[df.index[i], 'position'] = -1

        return df

    def backtest(self, data, initial_capital=100000):
        df = self.generate_unique_signals(data)

        # Initialize portfolio
        portfolio = pd.DataFrame(index=df.index)
        portfolio['position'] = df['position']
        portfolio['returns'] = df['returns']
        portfolio['strategy_returns'] = portfolio['position'].shift(1) * portfolio['returns']

        # Calculate cumulative returns
        portfolio['cumulative_returns'] = (1 + portfolio['strategy_returns'].fillna(0)).cumprod()
        portfolio['portfolio_value'] = initial_capital * portfolio['cumulative_returns']

        # Calculate performance metrics
        total_return = (portfolio['portfolio_value'].iloc[-1] / initial_capital - 1) * 100
        sharpe_ratio = np.sqrt(252) * portfolio['strategy_returns'].mean() / portfolio['strategy_returns'].std()
        max_drawdown = (portfolio['portfolio_value'] / portfolio['portfolio_value'].cummax() - 1).min() * 100

        return {
            'portfolio': portfolio,
            'total_return': total_return,
            'sharpe_ratio': sharpe_ratio,
            'max_drawdown': max_drawdown
        }

def mock_order_book_data():
    return pd.DataFrame({
        'buy_volume': np.random.rand(100),
        'sell_volume': np.random.rand(100)
    })

def mock_sentiment_data():
    return pd.DataFrame({
        'positive_mentions': np.random.randint(0, 100, size=100),
        'negative_mentions': np.random.randint(0, 100, size=100)
    })

def mock_on_chain_data():
    return pd.DataFrame({
        'large_transactions': np.random.randint(0, 1000, size=100),
        'threshold': 500
    })

def run_innovative_strategy():
    # Mocking data inputs
    order_book_data = mock_order_book_data()
    sentiment_data = mock_sentiment_data()
    on_chain_data = mock_on_chain_data()

    # Combine all mock data along with mock returns
    data = pd.concat([order_book_data, sentiment_data, on_chain_data], axis=1)
    data['returns'] = np.random.normal(0, 0.01, size=len(data))

    strategy = InnovativeCryptoStrategy()
    results = strategy.backtest(data)

    print(f"\nStrategy Results:")
    print(f"Total Return: {results['total_return']:.2f}%")
    print(f"Sharpe Ratio: {results['sharpe_ratio']:.2f}")
    print(f"Max Drawdown: {results['max_drawdown']:.2f}%")

    return results

results = run_innovative_strategy()



Strategy Results:
Total Return: 12.95%
Sharpe Ratio: 2.80
Max Drawdown: -4.88%
